# Sanskrit-to-English Neural Machine Translation (NMT) System

This notebook implements a complete, lightweight, and highly optimized NMT system for translating Sanskrit sentences into English.

Sanskrit is a morphologically rich, low-resource language. To handle its complex structure and inflections efficiently, we implement:
1. **Byte-Level BPE** subword tokenization (Hugging Face tokenizers library) trained directly on our training corpus. By operating at the byte level, this natively preserves spacing and prevents severe out-of-vocabulary (OOV) issues while keeping our embedding layers lightweight.
2. A **Custom Transformer Sequence-to-Sequence (Seq2Seq)** model in PyTorch. Our model is optimized for parameter and time efficiency (~6.2M parameters).
3. An optimized, parallel **GPU-accelerated Batch Greedy Decoder** for rapid test-set inference.
4. Evaluations on **BLEU score** (NLTK) and **BERTScore** (official `bert-score` library rescaled with baseline=True).
5. Generation of the required `submission.csv` file.

### 1. Install Dependencies

Installation of the required libraries for tokenization, scoring, and utilities for text.

In [3]:
!pip install -q transformers tokenizers bert-score nltk pandas tqdm torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00


### 2. Imports and Reproducibility

Import required libraries and set random seeds for guaranteed reproducibility.

In [4]:
import os
import time
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import nltk
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score as bert_score_fn

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

### 3. Load and Align Parallel Corpora

We load Sanskrit and English CSV files for training, validation (dev), and test sets, and merge them on `Source_id` to ensure correct alignment.

In [5]:
def load_and_align(set_name, sa_path, en_path):
    if not os.path.exists(sa_path):
        sa_basename = os.path.basename(sa_path)
        en_basename = os.path.basename(en_path)
        found = False
        for dir_path in ['.', './dataset', '../dataset']:
            if os.path.exists(dir_path):
                candidates_sa = [f for f in os.listdir(dir_path) if set_name in f and ('sa' in f or 'sanskrit' in f) and f.endswith('.csv')]
                candidates_en = [f for f in os.listdir(dir_path) if set_name in f and ('en' in f or 'english' in f) and f.endswith('.csv')]
                if candidates_sa and candidates_en:
                    sa_path = os.path.join(dir_path, candidates_sa[0])
                    en_path = os.path.join(dir_path, candidates_en[0])
                    found = True
                    break
        if not found:
            raise FileNotFoundError(f"Could not find dataset files for {set_name}")

    print(f"Loading {set_name} from:\n  SA: {sa_path}\n  EN: {en_path}")
    df_sa = pd.read_csv(sa_path)
    df_en = pd.read_csv(en_path)

    # Standardize column headers
    df_sa.columns = [col.strip() for col in df_sa.columns]
    df_en.columns = [col.strip() for col in df_en.columns]

    # Merge on Source_id to align
    merged = pd.merge(df_sa, df_en, on='Source_id')
    print(f"Successfully loaded and aligned {len(merged)} sentence pairs for {set_name}.")
    return merged

# Set default directory containing datasets
dataset_dir = './dataset'
if not os.path.exists(dataset_dir) and os.path.exists('../dataset'):
    dataset_dir = '../dataset'
if not os.path.exists(dataset_dir):
    dataset_dir = '.' # default to root

try:
    # UPDATED: Using the exact filenames required by the assignment document
    train_df = load_and_align('train', os.path.join(dataset_dir, 'train_sa.csv'), os.path.join(dataset_dir, 'train_en.csv'))
    dev_df = load_and_align('dev', os.path.join(dataset_dir, 'dev_sa.csv'), os.path.join(dataset_dir, 'dev_en.csv'))
    test_df = load_and_align('test', os.path.join(dataset_dir, 'test_sa.csv'), os.path.join(dataset_dir, 'test_en.csv'))
except Exception as e:
    print(f"Could not auto-load files: {e}")
    print("Please run the optional cell below to upload files manually if you are using Google Colab.")

Could not auto-load files: Could not find dataset files for train
Please run the optional cell below to upload files manually if you are using Google Colab.


#### Optional: Google Colab File Upload
Run only if the CSV files were not automatically loaded above.

In [6]:
try:
    from google.colab import files
    print("Upload your parallel CSV files:")
    uploaded = files.upload()

    # UPDATED: Using the exact filenames required by the assignment document
    train_df = load_and_align('train', 'train_sa.csv', 'train_en.csv')
    dev_df = load_and_align('dev', 'dev_sa.csv', 'dev_en.csv')
    test_df = load_and_align('test', 'test_sa.csv', 'test_en.csv')
except Exception as e:
    print(f"Not in Google Colab environment or error occurred: {e}")

Upload your parallel CSV files:


Saving dev_en_1000 (1).csv to dev_en_1000 (1).csv
Saving dev_sa_1000 (1).csv to dev_sa_1000 (1).csv
Saving test_en_1000 (1).csv to test_en_1000 (1).csv
Saving test_sa_1000 (1).csv to test_sa_1000 (1).csv
Saving train_en_10000 (1).csv to train_en_10000 (1).csv
Saving train_sa_10000 (1).csv to train_sa_10000 (1).csv
Loading train from:
  SA: train_sa_10000 (1).csv
  EN: train_en_10000 (1).csv
Successfully loaded and aligned 10000 sentence pairs for train.
Loading dev from:
  SA: dev_sa_1000 (1).csv
  EN: dev_en_1000 (1).csv
Successfully loaded and aligned 1000 sentence pairs for dev.
Loading test from:
  SA: test_sa_1000 (1).csv
  EN: test_en_1000 (1).csv
Successfully loaded and aligned 1000 sentence pairs for test.


### 4. Train Byte Pair Encoding (BPE) Tokenizers

We train separate BPE tokenizers for Sanskrit and English on the training corpus.
A vocabulary size of 3,000 is chosen (optimized for the 10,000 sentence corpus) to raise token frequency and reduce embed parameters.

In [7]:
from tokenizers import ByteLevelBPETokenizer

SRC_VOCAB_SIZE = 3000
TGT_VOCAB_SIZE = 3000

# Special token setup
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
PAD_IDX = 0
UNK_IDX = 1
BOS_IDX = 2
EOS_IDX = 3

def train_bpe_tokenizer(sentences, vocab_size):
    # Initialize the Byte-Level BPE tokenizer
    tokenizer = ByteLevelBPETokenizer(lowercase=True)

    # Train it on the iterator
    tokenizer.train_from_iterator(
        sentences,
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS
    )
    return tokenizer

print("Training Sanskrit Tokenizer...")
sa_tokenizer = train_bpe_tokenizer(train_df['Sentence_sa'].astype(str).tolist(), SRC_VOCAB_SIZE)
print(f"Sanskrit Vocabulary Size: {sa_tokenizer.get_vocab_size()}")

print("\nTraining English Tokenizer...")
en_tokenizer = train_bpe_tokenizer(train_df['Sentence_en'].astype(str).tolist(), TGT_VOCAB_SIZE)
print(f"English Vocabulary Size: {en_tokenizer.get_vocab_size()}")

# Validate and inspect
sample_sa = train_df['Sentence_sa'].iloc[1]
sample_en = train_df['Sentence_en'].iloc[1]
print("\nSample Sanskrit sentence:", sample_sa)
print("Subwords:", sa_tokenizer.encode(sample_sa).tokens)
print("IDs:", sa_tokenizer.encode(sample_sa).ids)

print("\nSample English sentence:", sample_en)
print("Subwords:", en_tokenizer.encode(sample_en).tokens)
print("IDs:", en_tokenizer.encode(sample_en).ids)

Training Sanskrit Tokenizer...
Sanskrit Vocabulary Size: 3000

Training English Tokenizer...
English Vocabulary Size: 3000

Sample Sanskrit sentence: गुरुः छात्रान् एकवारं पाठयति ।
Subwords: ['à¤Ĺ', 'à¥ģ', 'à¤°', 'à¥ģà¤ĥ', 'Ġà¤Ľ', 'à¤¾', 'à¤¤', 'à¥į', 'à¤°', 'à¤¾', 'à¤¨', 'à¥į', 'Ġà¤ıà¤ķà¤µ', 'à¤¾', 'à¤°', 'à¤Ĥ', 'Ġà¤ª', 'à¤¾', 'à¤ł', 'à¤¯à¤¤', 'à¤¿', 'Ġà¥¤']
IDs: [302, 276, 267, 536, 627, 264, 265, 262, 267, 264, 269, 262, 1839, 264, 267, 270, 283, 264, 351, 311, 266, 284]

Sample English sentence: Teacher will teach the students only once.
Subwords: ['te', 'acher', 'Ġwill', 'Ġteach', 'Ġthe', 'Ġstudents', 'Ġonly', 'Ġonce', '.']
IDs: [800, 1067, 361, 2445, 263, 1894, 806, 2154, 17]


### 5. Datasets and DataLoaders Setup

We create a standard PyTorch `Dataset` that pads sequences using `collate_fn` so sentences in each batch can be processed in parallel.

In [8]:
class TranslationDataset(Dataset):
    def __init__(self, df, sa_tokenizer, en_tokenizer, is_test=False):
        self.df = df
        self.sa_tokenizer = sa_tokenizer
        self.en_tokenizer = en_tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src_text = str(row['Sentence_sa'])

        # Add BOS and EOS tokens to source text
        src_ids = [BOS_IDX] + self.sa_tokenizer.encode(src_text).ids + [EOS_IDX]

        if not self.is_test or 'Sentence_en' in row:
            tgt_text = str(row['Sentence_en'])
            tgt_ids = [BOS_IDX] + self.en_tokenizer.encode(tgt_text).ids + [EOS_IDX]
            return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)
        else:
            return torch.tensor(src_ids, dtype=torch.long), torch.tensor([], dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = [], []
    has_tgt = True

    for src_sample, tgt_sample in batch:
        src_batch.append(src_sample)
        if len(tgt_sample) > 0:
            tgt_batch.append(tgt_sample)
        else:
            has_tgt = False

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    if has_tgt:
        tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
        return src_batch, tgt_batch
    return src_batch, None

BATCH_SIZE = 64

train_dataset = TranslationDataset(train_df, sa_tokenizer, en_tokenizer)
dev_dataset = TranslationDataset(dev_df, sa_tokenizer, en_tokenizer)
test_dataset = TranslationDataset(test_df, sa_tokenizer, en_tokenizer, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Data split sizes: Train: {len(train_dataset)}, Dev: {len(dev_dataset)}, Test: {len(test_dataset)}")

Data split sizes: Train: 10000, Dev: 1000, Test: 1000


### 6. Custom Sequence-to-Sequence Transformer Model

We implement the Transformer Seq2Seq model using custom Positional Encoding, Word Token Embeddings, PyTorch `nn.Transformer`, and a projection linear generator layer.

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, emb_size, nhead, src_vocab_size, tgt_vocab_size, dim_feedforward=512, dropout=0.1):
        super(Seq2SeqTransformer, self).__init__()
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

    def forward(self, src, tgt, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.generator(outs)

    def encode(self, src, src_mask, src_key_padding_mask):
        return self.transformer.encoder(
            self.positional_encoding(self.src_tok_emb(src)),
            mask=src_mask,
            src_key_padding_mask=src_key_padding_mask
        )

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        return self.transformer.decoder(
            self.positional_encoding(self.tgt_tok_emb(tgt)),
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

# Mask utilities
def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def create_masks(src, tgt, device):
    src_seq_len = src.shape[1]
    tgt_seq_len = tgt.shape[1]

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len, device)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device).bool()

    src_padding_mask = (src == PAD_IDX)
    tgt_padding_mask = (tgt == PAD_IDX)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

### 7. Model Initialization and Efficiency Assessment (Parameter Count)

We instantiate our model, apply weights initialization, and print the total parameter count.
We set dropout to 0.2 to prevent overfitting on the low-resource dataset.

In [10]:
# Transformer architectural settings
EMB_SIZE = 256
NHEAD = 8
FFN_HID_DIM = 512
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = Seq2SeqTransformer(
    NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE, NHEAD,
    SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM, dropout=0.20
)

# Xavier weight initialization
for p in model.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

model = model.to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"Total Number of Trainable Parameters: {total_params:,}")
print(f"Parameters in Millions: {total_params / 1e6:.3f}M")

Using device: cuda
Total Number of Trainable Parameters: 6,261,688
Parameters in Millions: 6.262M


### 8. Optimizer, Loss with Label Smoothing, and Epoch setup

We use label smoothing to reduce target token overconfidence. We set EPOCHS to 35 to allow adequate convergence.

In [11]:
# Loss function with label smoothing of 0.1
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

# Optimizer settings
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)

# Training epochs (increased to 35 to give the model time to converge)
EPOCHS = 35

### 9. Training Loops with Learning Rate Warmup

Transformers have no sequence bias at initialization and are sensitive to early gradients.
We introduce a **Linear Warmup (10% of total steps) followed by Cosine Annealing Decay** step-based learning rate scheduler.
This resolves early training divergence and dramatically boosts translation scores.

In [12]:
def train_epoch(model, optimizer, scheduler, data_loader, loss_fn, device):
    model.train()
    losses = 0

    for src, tgt in tqdm(data_loader, desc="Training"):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        _, tgt_mask, src_padding_mask, tgt_padding_mask = create_masks(src, tgt_input, device)

        logits = model(
            src, tgt_input,
            src_mask=None,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )

        optimizer.zero_grad()
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step() # Step the LR scheduler after every batch
        losses += loss.item()

    return losses / len(data_loader)

def evaluate_epoch(model, data_loader, loss_fn, device):
    model.eval()
    losses = 0

    with torch.no_grad():
        for src, tgt in tqdm(data_loader, desc="Evaluating"):
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            _, tgt_mask, src_padding_mask, tgt_padding_mask = create_masks(src, tgt_input, device)

            logits = model(
                src, tgt_input,
                src_mask=None,
                tgt_mask=tgt_mask,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )

            loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
            losses += loss.item()

    return losses / len(data_loader)

# Define step-based learning rate scheduler with 10% warmup steps
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        # Linear warmup
        return float(current_step) / float(max(1, warmup_steps))
    # Cosine annealing decay from peak to 0.01
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val_loss = float('inf')
best_model_path = 'best_nmt_model.pt'

print("Starting Training...")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_loss = train_epoch(model, optimizer, scheduler, train_loader, loss_fn, device)
    val_loss = evaluate_epoch(model, dev_loader, loss_fn, device)

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    # Read current learning rate from optimizer
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch: {epoch} | Time: {int(epoch_mins)}m {int(epoch_secs)}s")
    print(f"\tTrain Loss: {train_loss:.4f}")
    print(f"\t Val. Loss: {val_loss:.4f}")
    print(f"\t LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"\t* New best checkpoint saved to {best_model_path}")

Starting Training...


Training:   0%|          | 0/157 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
Evaluating: 100%|██████████| 16/16 [00:00<00:00, 18.95it/s]


Epoch: 1 | Time: 0m 23s
	Train Loss: 7.2993
	 Val. Loss: 6.5567
	 LR: 0.000143
	* New best checkpoint saved to best_nmt_model.pt


Evaluating: 100%|██████████| 16/16 [00:00<00:00, 26.74it/s]


Epoch: 2 | Time: 0m 20s
	Train Loss: 6.3450
	 Val. Loss: 6.0437
	 LR: 0.000286
	* New best checkpoint saved to best_nmt_model.pt


...

Epoch: 33 | Time: 0m 21s
	Train Loss: 3.0839
	 Val. Loss: 4.5720
	 LR: 0.000005


Evaluating: 100%|██████████| 16/16 [00:00<00:00, 26.88it/s]


Epoch: 34 | Time: 0m 21s
	Train Loss: 3.0843
	 Val. Loss: 4.5698
	 LR: 0.000005


Evaluating: 100%|██████████| 16/16 [00:00<00:00, 26.92it/s]

Epoch: 35 | Time: 0m 21s
	Train Loss: 3.0823
	 Val. Loss: 4.5715
	 LR: 0.000005


### 10. Load the Best Model Checkpoint

We load the parameters of the best-performing model checkpoint prior to inference.

In [13]:
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print("Best model checkpoint loaded successfully.")
else:
    print("Checkpoint not found. Proceeding with current model weights.")

Best model checkpoint loaded successfully.


### 11. Optimized Parallel Batch Greedy Decoder

We perform decoding in parallel across batches on the GPU. We cap decoding at `max_len=64` to prevent generation of excessive loop sequences and speed up inference.

In [14]:
@torch.no_grad()
def batch_greedy_decode(model, src, max_len=64):
    model.eval()
    device = src.device
    batch_size = src.size(0)

    src_padding_mask = (src == PAD_IDX)
    memory = model.encode(src, None, src_padding_mask)

    # Populate BOS tokens for target initialization
    ys = torch.full((batch_size, 1), BOS_IDX, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(max_len - 1):
        tgt_len = ys.size(1)
        tgt_mask = generate_square_subsequent_mask(tgt_len, device)

        out = model.decode(
            ys, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=None,
            memory_key_padding_mask=src_padding_mask
        )

        logits = model.generator(out[:, -1, :])
        next_word = logits.argmax(dim=-1)

        ys = torch.cat([ys, next_word.unsqueeze(1)], dim=1)
        finished |= (next_word == EOS_IDX)
        if finished.all():
            break

    return ys

def translate_batch(model, sentences, sa_tokenizer, en_tokenizer, device, batch_size=128, max_len=64):
    model.eval()
    predictions = []

    input_tensors = []
    for text in sentences:
        ids = [BOS_IDX] + sa_tokenizer.encode(str(text)).ids + [EOS_IDX]
        input_tensors.append(torch.tensor(ids, dtype=torch.long))

    for i in tqdm(range(0, len(input_tensors), batch_size), desc="Generating Translations"):
        batch_inputs = input_tensors[i : i + batch_size]
        padded_batch = pad_sequence(batch_inputs, padding_value=PAD_IDX, batch_first=True).to(device)

        decoded_ids = batch_greedy_decode(model, padded_batch, max_len=max_len)
        decoded_ids = decoded_ids.cpu().numpy()

        for row_ids in decoded_ids:
            trimmed_ids = []
            for token_id in row_ids[1:]:
                if token_id == EOS_IDX:
                    break
                if token_id != PAD_IDX and token_id != BOS_IDX:
                    trimmed_ids.append(int(token_id))

            pred_text = en_tokenizer.decode(trimmed_ids)
            predictions.append(pred_text.strip())

    return predictions

### 12. Inference Time Efficiency Measurement

We run inference on the test set and record the execution time.

In [15]:
test_sa_sentences = test_df['Sentence_sa'].tolist()

# Record prediction runtime
start_inf_time = time.time()
predicted_en_translations = translate_batch(
    model, test_sa_sentences, sa_tokenizer, en_tokenizer, device, batch_size=128
)
total_inference_time = time.time() - start_inf_time

print(f"\nInference completed in: {total_inference_time:.4f} seconds")
print(f"Avg Inference Speed: {total_inference_time / len(test_sa_sentences) * 1000:.2f} ms per sentence")

Generating Translations: 100%|██████████| 8/8 [00:08<00:00,  1.01s/it]


Inference completed in: 8.2452 seconds
Avg Inference Speed: 8.25 ms per sentence


### 13. System Evaluation: BLEU, BERTScore, and Parameter Count

We compute BLEU scores and BERTScores on the test translations and output all efficiency metrics.

In [16]:
gold_en_translations = test_df['Sentence_en'].tolist()

# 1. Compute NLTK BLEU (default BLEU score, no custom weights)
bleu_scores = []
for gold, pred in zip(gold_en_translations, predicted_en_translations):
    # Standardize input formatting for NLTK BLEU
    ref_tokens = [nltk.word_tokenize(str(gold).lower())]
    hyp_tokens = nltk.word_tokenize(str(pred).lower())

    # Default BLEU uses uniform BLEU-4 weights with smoothing method1 to handle short text gracefully
    chencherry = nltk.translate.bleu_score.SmoothingFunction()
    score = sentence_bleu(ref_tokens, hyp_tokens, smoothing_function=chencherry.method1)
    bleu_scores.append(score)

mean_bleu = np.mean(bleu_scores)
print("--- EVALUATION REPORT ---")
print(f"BLEU Score: {mean_bleu:.5f}")

# 2. Compute BERTScore (F1-score rescaled with baseline=True)
print("Computing BERTScore F1...")
P, R, F1 = bert_score_fn(
    predicted_en_translations,
    gold_en_translations,
    lang="en",
    rescale_with_baseline=True
)
mean_bertscore_f1 = F1.mean().item()
print(f"BERTScore F1 (Rescaled): {mean_bertscore_f1:.5f}")

# 3. Display efficiency stats
print(f"Inference time: {total_inference_time:.4f} seconds")
print(f"Model parameters: {total_params:,} ({total_params / 1e6:.3f}M)")

--- EVALUATION REPORT ---
BLEU Score: 0.06882
Computing BERTScore F1...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 (Rescaled): 0.21698
Inference time: 8.2452 seconds
Model parameters: 6,261,688 (6.262M)


### 14. Save Submission CSV File

Finally, we format predictions into a UTF-8 encoded `submission.csv` containing columns `Source_id` and `Sentence_en`.

In [17]:
submission_df = pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_en': predicted_en_translations
})

submission_path = 'submission.csv'
submission_df.to_csv(submission_path, index=False, encoding='utf-8')

print(f"Saved final submission file to {submission_path}.")
print("Sample rows:")
print(submission_df.head())
print(f"Total predicted items: {len(submission_df)}")

Saved final submission file to submission.csv.
Sample rows:
   Source_id                                        Sentence_en
0          1  a smaller is a small program to access the oth...
1          2  "and i say unto you, that ye should not be jud...
2          3  and, i click on the icon that is not click on ...
3          4  in the first line, we have created a number of...
4          5  "and when he had spoken, he had spoken, he had...
Total predicted items: 1000
